# **1. Data collection**

This notebook collects and incrementally updates player-level match minutes by season and round across 16 domestic leagues. All listed players are retained at collection time, regardless of age, position, or match participation.

Teams, roster memberships, player profiles, and match-level minutes are stored as separate raw tables. HTTP responses are cached in `data/raw/cache/`, while resumable checkpoints and a collection-run audit are written to `data/raw/*.parquet`.

In [ ]:
"""Resumable Transfermarkt collection of player match minutes.

The collector stores teams, roster memberships, player profiles, and one
observation per player and match. Raw HTTP responses are cached locally.
"""

from __future__ import annotations

from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import re
import time
from typing import Any
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup


ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

BASE_URL = "https://www.transfermarkt.com"
API_URL = "https://tmapi.transfermarkt.technology"
SEASONS = range(2019, 2027)
LEAGUE_CODES = [
    "ENG-1", "SPA-1", "GER-1", "ITA-1", "FRA-1", "POR-1", "BR-1",
    "NED-1", "BEL-1", "ARG-1", "COL-1", "PAR-1", "URU-1", "KSA-1",
    "MEX-1", "USA-1",
]
LEAGUE_CATALOG = {
    "ENG-1": {"name": "Premier League", "sources": [("GB1", "premier-league")]},
    "SPA-1": {"name": "LaLiga", "sources": [("ES1", "laliga")]},
    "GER-1": {"name": "Bundesliga", "sources": [("L1", "bundesliga")]},
    "ITA-1": {"name": "Serie A", "sources": [("IT1", "serie-a")]},
    "FRA-1": {"name": "Ligue 1", "sources": [("FR1", "ligue-1")]},
    "POR-1": {"name": "Liga Portugal", "sources": [("PO1", "liga-portugal")]},
    "BR-1": {"name": "Brasileirão", "sources": [("BRA1", "campeonato-brasileiro-serie-a")]},
    "NED-1": {"name": "Eredivisie", "sources": [("NL1", "eredivisie")]},
    "BEL-1": {"name": "Jupiler Pro League", "sources": [("BE1", "jupiler-pro-league")]},
    "ARG-1": {
        "name": "Liga Profesional",
        "sources": [("AR1N", "liga-profesional-de-futbol"),
                    ("ARG1", "torneo-apertura"), ("ARGC", "torneo-clausura")],
    },
    "COL-1": {
        "name": "Primera A",
        "sources": [("COLP", "liga-dimayor-apertura"),
                    ("COL1", "liga-dimayor-finalizacion"),
                    ("COLF", "fase-final-liga-dimayor-ii")],
    },
    "PAR-1": {
        "name": "Primera División",
        "sources": [("PR1A", "primera-division-apertura"),
                    ("PR1C", "primera-division-clausura")],
    },
    "URU-1": {
        "name": "Primera División",
        "sources": [("URU1", "primera-division-apertura"),
                    ("URUI", "torneo-intermedio"),
                    ("URUC", "primera-division-clausura")],
    },
    "KSA-1": {"name": "Saudi Pro League", "sources": [("SA1", "saudi-pro-league")]},
    "MEX-1": {
        "name": "Liga MX",
        "sources": [("MEXA", "liga-mx-apertura"), ("MEX1", "liga-mx-clausura")],
    },
    "USA-1": {"name": "Major League Soccer", "sources": [("MLS1", "major-league-soccer")]},
}
LEAGUES = [
    {
        "competition_code": code,
        "competition_name": LEAGUE_CATALOG[code]["name"],
        "source_competition_id": source_id,
        "slug": slug,
    }
    for code in LEAGUE_CODES
    for source_id, slug in LEAGUE_CATALOG[code]["sources"]
]
SOURCE_TO_CODE = {
    league["source_competition_id"]: league["competition_code"] for league in LEAGUES
}
SOURCE_SEASONS = {
    'AR1N': range(2019, 2025),
    'ARG1': range(2024, 2026),
    'ARGC': range(2024, 2026),
}
REQUEST_DELAY = 1.0
REQUEST_TIMEOUT = 30
CHECKPOINT_EVERY = 500
REFRESH_AFTER_HOURS = 24
ACTIVE_SEASON_LOOKBACK = 1
PARSER_VERSION = "5-domestic-league-match-minutes"
COLLECTION_SCOPE = f"minutes-{min(SEASONS)}-{max(SEASONS)}-{len(LEAGUE_CODES)}-leagues-v3"

TEAM_RE = re.compile(
    r"^/(?P<slug>[^/]+)/kader/verein/(?P<team_id>\d+)/saison_id/(?P<season_id>\d+)"
)
PLAYER_RE = re.compile(r"^/(?P<slug>[^/]+)/profil/spieler/(?P<player_id>\d+)")


def _project_root() -> Path:
    return ROOT


def _obj(value: Any) -> dict[str, Any]:
    return value if isinstance(value, dict) else {}


def _standardize_raw_schema(
    teams: pd.DataFrame, matches: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Upgrade previous raw tables without recollecting completed observations."""

    teams = teams.copy()
    matches = matches.copy()
    if "source_competition_id" not in teams and "competition_id" in teams:
        teams = teams.rename(columns={"competition_id": "source_competition_id"})
    if not teams.empty:
        teams["source_competition_id"] = teams["source_competition_id"].astype(str)
        teams["competition_code"] = teams["source_competition_id"].map(SOURCE_TO_CODE)
        teams = teams.loc[teams["competition_code"].notna()].copy()

    if "source_competition_id" not in matches and "competition_id" in matches:
        matches = matches.rename(columns={"competition_id": "source_competition_id"})
    if not matches.empty:
        matches["source_competition_id"] = matches["source_competition_id"].astype(str)
        matches["competition_code"] = matches["source_competition_id"].map(SOURCE_TO_CODE)
        matches = matches.loc[matches["competition_code"].notna()].copy()
    return teams, matches


class TransfermarktCollector:
    """Collect Transfermarkt data with local caching and Parquet checkpoints."""

    def __init__(
        self,
        root: Path | str | None = None,
        request_delay: float = REQUEST_DELAY,
        request_timeout: int = REQUEST_TIMEOUT,
        checkpoint_every: int = CHECKPOINT_EVERY,
    ) -> None:
        self.root = Path(root).resolve() if root else _project_root()
        self.raw = self.root / "data" / "raw"
        self.cache = self.raw / "cache"
        self.raw.mkdir(parents=True, exist_ok=True)
        self.cache.mkdir(parents=True, exist_ok=True)
        self.request_delay = request_delay
        self.request_timeout = request_timeout
        self.checkpoint_every = checkpoint_every
        self.session = requests.Session()
        self.session.headers.update(
            {
                "User-Agent": "football-data-research/0.1 (academic research)",
                "Accept": "application/json, text/html;q=0.9, */*;q=0.8",
            }
        )

    def cache_path(self, url: str, suffix: str) -> Path:
        label = re.sub(r"[^a-zA-Z0-9]+", "-", urlparse(url).path).strip("-")[-80:]
        digest = hashlib.sha256(url.encode()).hexdigest()[:16]
        return self.cache / f"{label}-{digest}.{suffix}"

    def fetch(self, url: str, kind: str = "text", force: bool = False) -> Any:
        suffix = "json" if kind == "json" else "html"
        path = self.cache_path(url, suffix)
        if path.exists() and not force:
            return json.loads(path.read_text()) if kind == "json" else path.read_text()

        last_error: Exception | None = None
        for attempt in range(3):
            try:
                time.sleep(self.request_delay)
                response = self.session.get(url, timeout=self.request_timeout)
                response.raise_for_status()
                value = response.json() if kind == "json" else response.text
                temporary = path.with_suffix(path.suffix + ".tmp")
                serialized = json.dumps(value, ensure_ascii=False) if kind == "json" else value
                temporary.write_text(serialized)
                temporary.replace(path)
                return value
            except Exception as exc:
                last_error = exc
                time.sleep(2**attempt)
        raise RuntimeError(f"Failed to collect {url}") from last_error

    def cache_is_stale(
        self, url: str, kind: str, max_age_hours: float
    ) -> bool:
        suffix = "json" if kind == "json" else "html"
        path = self.cache_path(url, suffix)
        if not path.exists():
            return True
        age_seconds = time.time() - path.stat().st_mtime
        return age_seconds >= max_age_hours * 3_600

    def cache_timestamp(self, url: str, kind: str) -> str | None:
        suffix = "json" if kind == "json" else "html"
        path = self.cache_path(url, suffix)
        if not path.exists():
            return None
        return datetime.fromtimestamp(path.stat().st_mtime, timezone.utc).isoformat()

    def load(self, name: str) -> pd.DataFrame:
        path = self.raw / name
        return pd.read_parquet(path) if path.exists() else pd.DataFrame()

    def save(self, frame: pd.DataFrame, name: str) -> None:
        path = self.raw / name
        temporary = path.with_suffix(".parquet.tmp")
        frame.to_parquet(temporary, index=False)
        temporary.replace(path)

    @staticmethod
    def upsert(existing: pd.DataFrame, new: pd.DataFrame, keys: list[str]) -> pd.DataFrame:
        frames = [frame for frame in (existing, new) if not frame.empty]
        if not frames:
            return pd.DataFrame()
        return pd.concat(frames, ignore_index=True).drop_duplicates(keys, keep="last")

    @staticmethod
    def parse_teams(html: str, league: dict[str, Any], season_id: int) -> pd.DataFrame:
        rows = []
        for anchor in BeautifulSoup(html, "html.parser").select('a[href*="/kader/verein/"]'):
            href = str(anchor.get("href", ""))
            match = TEAM_RE.match(urlparse(href).path)
            if match and int(match["season_id"]) == season_id:
                rows.append(
                    {
                        "competition_code": league["competition_code"],
                        "source_competition_id": league["source_competition_id"],
                        "competition_name": league["competition_name"],
                        "season_id": season_id,
                        "team_id": match["team_id"],
                        "team_name": " ".join(
                            (anchor.get("title") or anchor.get_text(" ", strip=True)).split()
                        ),
                        "team_url": urljoin(BASE_URL, href),
                    }
                )
        columns = [
            "competition_code", "source_competition_id", "competition_name",
            "season_id", "team_id", "team_name", "team_url",
        ]
        return pd.DataFrame(rows, columns=columns).drop_duplicates(
            ["source_competition_id", "season_id", "team_id"]
        )

    @staticmethod
    def parse_roster(
        html: str, competition_id: str, season_id: int, team_id: str
    ) -> pd.DataFrame:
        rows = []
        for tr in BeautifulSoup(html, "html.parser").select("tr.odd, tr.even"):
            anchor = tr.select_one('td.posrela td.hauptlink a[href*="/profil/spieler/"]')
            if anchor is None:
                continue
            match = PLAYER_RE.match(urlparse(str(anchor.get("href", ""))).path)
            if not match:
                continue
            position = tr.select_one("td.rueckennummer[title]")
            rows.append(
                {
                    "source_competition_id": competition_id,
                    "source_season_id": season_id,
                    "source_team_id": str(team_id),
                    "player_id": match["player_id"],
                    "player_name": " ".join(anchor.get_text(" ", strip=True).split()),
                    "player_position": position.get("title", "") if position else "",
                }
            )
        result = pd.DataFrame(rows).drop_duplicates(
            ["source_competition_id", "source_season_id", "source_team_id", "player_id"]
        )
        if result.empty:
            raise ValueError(f"No players found: {team_id} {season_id}")
        return result

    @staticmethod
    def parse_profile(payload: dict[str, Any], player_id: str) -> dict[str, Any]:
        data = _obj(payload.get("data", payload))
        return {
            "player_id": str(player_id),
            "birth_date": _obj(data.get("lifeDates")).get("dateOfBirth"),
            "profile_collected_at": datetime.now(timezone.utc).isoformat(),
            "parser_version": PARSER_VERSION,
        }

    @staticmethod
    def parse_matches(payload: dict[str, Any], player_id: str) -> pd.DataFrame:
        collected_at = datetime.now(timezone.utc).isoformat()
        rows = []
        for event in _obj(payload.get("data", payload)).get("performance", []):
            game = _obj(event.get("gameInformation"))
            clubs = _obj(event.get("clubsInformation"))
            club = _obj(clubs.get("club"))
            opponent = _obj(clubs.get("opponent"))
            stats = _obj(event.get("statistics"))
            general = _obj(stats.get("generalStatistics"))
            playing = _obj(stats.get("playingTimeStatistics"))
            minutes = playing.get("playedMinutes")
            source_id = str(game.get("competitionId"))
            season = _obj(game.get("season"))
            rows.append(
                {
                    "player_id": str(player_id),
                    "game_id": game.get("gameId"),
                    "competition_code": SOURCE_TO_CODE.get(source_id),
                    "source_competition_id": source_id,
                    "season_id": game.get("seasonId"),
                    "season_display": season.get("display"),
                    "season_cyclical_name": season.get("cyclicalName"),
                    "season_non_cyclical_name": season.get("nonCyclicalName"),
                    "round_id": game.get("gameDay"),
                    "match_date_utc": _obj(game.get("date")).get("dateTimeUTC"),
                    "match_duration": game.get("gameDuration"),
                    "club_id": club.get("clubId"),
                    "opponent_club_id": opponent.get("clubId"),
                    "venue": club.get("venue"),
                    "participation_state": general.get("participationState"),
                    "position_id": general.get("positionId"),
                    "is_starter": playing.get("isStarting"),
                    "minutes_played": 0 if minutes is None else minutes,
                    "injury_id": general.get("injuryId"),
                    "absence_id": general.get("absenceId"),
                    "game_state": game.get("gameState"),
                    "is_live_game": game.get("isLiveGame"),
                    "match_collected_at": collected_at,
                    "parser_version": PARSER_VERSION,
                }
            )
        return pd.DataFrame(rows)

    def collect(
        self,
        mode: str = "update",
        force: bool = False,
        refresh_after_hours: float = REFRESH_AFTER_HOURS,
        max_players: int | None = None,
    ) -> dict[str, pd.DataFrame]:
        if mode not in {"backfill", "update"}:
            raise ValueError("mode must be 'backfill' or 'update'")

        started_at = datetime.now(timezone.utc)
        run_id = started_at.strftime("%Y%m%dT%H%M%S%fZ")
        teams = self._collect_teams(force, mode, refresh_after_hours)
        rosters = self._collect_rosters(teams, force, mode, refresh_after_hours)
        players = self.load("players.parquet")
        matches = self.load("player_matches.parquet")
        status = self.load("collection_status.parquet")
        teams, matches = _standardize_raw_schema(teams, matches)
        self.save(teams, "teams.parquet")
        if not matches.empty:
            self.save(matches, "player_matches.parquet")

        all_player_ids = rosters["player_id"].astype(str).drop_duplicates().tolist()
        active_player_ids = _active_players(rosters)
        successful = _successful_players(status)
        status_lookup = (
            status.drop_duplicates("player_id", keep="last")
            .assign(player_id=lambda x: x.player_id.astype(str))
            .set_index("player_id").to_dict("index")
            if not status.empty else {}
        )
        target_horizon = max(SEASONS)
        backfill_pending = set(all_player_ids) - successful
        horizon_pending: set[str] = set()
        stale_active: set[str] = set()
        if mode == "update":
            for player_id in active_player_ids:
                previous = status_lookup.get(player_id, {})
                processed = pd.to_numeric(
                    previous.get("processed_through_season_id"), errors="coerce"
                )
                scope_changed = previous.get("collection_scope") != COLLECTION_SCOPE
                if scope_changed or pd.isna(processed) or int(processed) < target_horizon:
                    horizon_pending.add(player_id)
                performance_url = f"{API_URL}/player/{player_id}/performance-game"
                if self.cache_is_stale(performance_url, "json", refresh_after_hours):
                    stale_active.add(player_id)

        selected = backfill_pending | horizon_pending | stale_active
        if force and mode == "update":
            selected |= active_player_ids
        pending = [player_id for player_id in all_player_ids if player_id in selected]
        pending = pending if max_players is None else pending[:max_players]

        known_profiles = set(players.get("player_id", pd.Series(dtype=str)).astype(str))
        matches_before = len(matches)
        network_refreshes = 0
        cache_reparses = 0
        profiles_collected = 0
        player_errors = 0
        new_players: list[dict[str, Any]] = []
        new_matches: list[pd.DataFrame] = []
        new_status: list[dict[str, Any]] = []

        def checkpoint() -> None:
            nonlocal players, matches, status
            nonlocal new_players, new_matches, new_status

            players = self.upsert(players, pd.DataFrame(new_players), ["player_id"])
            incoming_matches = (
                pd.concat(new_matches, ignore_index=True) if new_matches else pd.DataFrame()
            )
            matches = self.upsert(matches, incoming_matches, ["player_id", "game_id"])
            status = self.upsert(status, pd.DataFrame(new_status), ["player_id"])

            if not players.empty:
                self.save(players, "players.parquet")
            if not matches.empty:
                self.save(matches, "player_matches.parquet")
            if not status.empty:
                self.save(status, "collection_status.parquet")
            new_players, new_matches, new_status = [], [], []

        target_competitions = set(SOURCE_TO_CODE)
        target_seasons = set(SEASONS)

        for number, player_id in enumerate(pending, 1):
            previous = status_lookup.get(player_id, {})
            performance_url = f"{API_URL}/player/{player_id}/performance-game"
            performance_was_cached = self.cache_path(
                performance_url, "json"
            ).exists()
            refresh_from_network = force or (
                mode == "update" and player_id in stale_active
            )
            try:
                if player_id not in known_profiles:
                    profile_payload = self.fetch(f"{API_URL}/player/{player_id}", "json")
                    new_players.append(self.parse_profile(profile_payload, player_id))
                    known_profiles.add(player_id)
                    profiles_collected += 1
                performance_payload = self.fetch(
                    performance_url, "json", refresh_from_network
                )
                used_network = refresh_from_network or not performance_was_cached
                network_refreshes += int(used_network)
                cache_reparses += int(not used_network)
                parsed_matches = self.parse_matches(performance_payload, player_id)
                if not parsed_matches.empty:
                    parsed_matches["season_id"] = pd.to_numeric(
                        parsed_matches["season_id"], errors="coerce"
                    ).astype("Int64")
                    parsed_matches = parsed_matches.loc[
                        parsed_matches["source_competition_id"].isin(target_competitions)
                        & parsed_matches["season_id"].isin(target_seasons)
                    ]
                    new_matches.append(parsed_matches)
                new_status.append(
                    {
                        "player_id": player_id,
                        "status": "ok",
                        "n_matches": len(parsed_matches),
                        "collected_at": datetime.now(timezone.utc).isoformat(),
                        "last_performance_refresh_at": self.cache_timestamp(
                            performance_url, "json"
                        ),
                        "processed_through_season_id": target_horizon,
                        "collection_scope": COLLECTION_SCOPE,
                        "collection_mode": (
                            "network_refresh" if refresh_from_network
                            else "network_download" if not performance_was_cached
                            else "cache_reparse"
                        ),
                        "error": None,
                    }
                )
            except Exception as exc:
                player_errors += 1
                new_status.append(
                    {
                        "player_id": player_id,
                        "status": "error",
                        "n_matches": previous.get("n_matches", 0),
                        "collected_at": datetime.now(timezone.utc).isoformat(),
                        "last_performance_refresh_at": previous.get(
                            "last_performance_refresh_at"
                        ),
                        "processed_through_season_id": previous.get(
                            "processed_through_season_id"
                        ),
                        "collection_scope": COLLECTION_SCOPE,
                        "collection_mode": "error",
                        "error": f"{type(exc).__name__}: {exc}",
                    }
                )
            if number % self.checkpoint_every == 0:
                checkpoint()

        if new_players or new_matches or new_status:
            checkpoint()

        finished_at = datetime.now(timezone.utc)
        run_record = {
            "run_id": run_id,
            "mode": mode,
            "started_at": started_at.isoformat(),
            "finished_at": finished_at.isoformat(),
            "collection_scope": COLLECTION_SCOPE,
            "season_id_min": min(SEASONS),
            "season_id_max": max(SEASONS),
            "roster_players": len(all_player_ids),
            "active_players": len(active_player_ids),
            "selected_players": len(pending),
            "network_refreshes": network_refreshes,
            "cache_reparses": cache_reparses,
            "profiles_collected": profiles_collected,
            "player_errors": player_errors,
            "match_rows_before": matches_before,
            "match_rows_after": len(matches),
            "match_rows_added": len(matches) - matches_before,
        }
        runs = self.upsert(
            self.load("collection_runs.parquet"), pd.DataFrame([run_record]), ["run_id"]
        )
        self.save(runs, "collection_runs.parquet")

        return {
            "teams": teams,
            "rosters": rosters,
            "players": players,
            "matches": matches,
            "status": status,
            "runs": runs,
        }

    def _collect_teams(
        self, force: bool, mode: str, refresh_after_hours: float
    ) -> pd.DataFrame:
        teams, _ = _standardize_raw_schema(self.load("teams.parquet"), pd.DataFrame())
        done = set(zip(teams.get("source_competition_id", []), teams.get("season_id", [])))
        for league in LEAGUES:
            source_id = league["source_competition_id"]
            for season_id in SOURCE_SEASONS.get(source_id, SEASONS):
                key = (league["source_competition_id"], season_id)
                url = (
                    f"{BASE_URL}/{league['slug']}/startseite/wettbewerb/"
                    f"{league['source_competition_id']}/plus/?saison_id={season_id}"
                )
                refresh_current = force or (
                    mode == "update"
                    and season_id >= max(SEASONS) - ACTIVE_SEASON_LOOKBACK
                    and self.cache_is_stale(url, "text", refresh_after_hours)
                )
                if key in done and not refresh_current:
                    continue
                try:
                    parsed = self.parse_teams(
                        self.fetch(url, force=refresh_current), league, season_id
                    )
                except Exception as exc:
                    print(f"Skipped {key}: {type(exc).__name__}: {exc}")
                    continue
                if parsed.empty:
                    print(f"Skipped {key}: no teams found")
                    continue
                teams = self.upsert(
                    teams, parsed, ["source_competition_id", "season_id", "team_id"]
                )
                self.save(teams, "teams.parquet")
                done.add(key)
        return teams

    def _collect_rosters(
        self,
        teams: pd.DataFrame,
        force: bool,
        mode: str,
        refresh_after_hours: float,
    ) -> pd.DataFrame:
        rosters = self.load("roster_memberships.parquet")
        if not rosters.empty:
            rosters["source_competition_id"] = rosters["source_competition_id"].astype(str)
            rosters = rosters.loc[
                rosters["source_competition_id"].isin(SOURCE_TO_CODE)
            ].copy()
            self.save(rosters, "roster_memberships.parquet")
        done = set(
            zip(
                rosters.get("source_competition_id", []),
                rosters.get("source_season_id", []),
                rosters.get("source_team_id", []),
            )
        )
        for team in teams.itertuples(index=False):
            key = (
                str(team.source_competition_id), int(team.season_id), str(team.team_id)
            )
            refresh_current = force or (
                mode == "update"
                and int(team.season_id) >= max(SEASONS) - ACTIVE_SEASON_LOOKBACK
                and self.cache_is_stale(team.team_url, "text", refresh_after_hours)
            )
            if key in done and not refresh_current:
                continue
            try:
                parsed = self.parse_roster(
                    self.fetch(team.team_url, force=refresh_current), *key
                )
            except Exception as exc:
                print(f"Skipped roster {key}: {type(exc).__name__}: {exc}")
                continue
            rosters = self.upsert(
                rosters,
                parsed,
                ["source_competition_id", "source_season_id", "source_team_id", "player_id"],
            )
            self.save(rosters, "roster_memberships.parquet")
            done.add(key)
        return rosters


def _successful_players(status: pd.DataFrame) -> set[str]:
    if status.empty:
        return set()
    state = status["status"].eq("ok")
    return set(status.loc[state, "player_id"].astype(str))


def _active_players(rosters: pd.DataFrame) -> set[str]:
    if rosters.empty:
        return set()
    frame = rosters.copy()
    frame["source_season_id"] = pd.to_numeric(
        frame["source_season_id"], errors="coerce"
    ).astype("Int64")
    latest = frame.groupby("source_competition_id")["source_season_id"].transform("max")
    active_floor = max(SEASONS) - ACTIVE_SEASON_LOOKBACK
    active = frame[frame["source_season_id"].eq(latest) & latest.ge(active_floor)]
    return set(active["player_id"].astype(str))


def collect(
    mode: str = "update",
    force: bool = False,
    refresh_after_hours: float = REFRESH_AFTER_HOURS,
    max_players: int | None = None,
    root: Path | str | None = None,
    request_delay: float = REQUEST_DELAY,
) -> dict[str, pd.DataFrame]:
    """Convenience entry point used by the notebook."""

    collector = TransfermarktCollector(root=root, request_delay=request_delay)
    return collector.collect(
        mode=mode, force=force, refresh_after_hours=refresh_after_hours,
        max_players=max_players,
    )


## **2. Competition codes**

Analysis uses stable codes such as `ENG-1`, `BR-1`, and `MEX-1`. Native Transfermarkt identifiers remain in `source_competition_id`; Apertura, Clausura, and other official stages of the domestic first division are combined under one analytical league code. Domestic cups and continental competitions are outside the collection scope.

## **3. Configuration sanity checks**

Deterministic checks verify that the league catalog is complete, source identifiers are unique, and split-season competitions map to a single analytical code.

In [ ]:
assert len(LEAGUE_CODES) == 16
assert set(LEAGUE_CODES) == set(LEAGUE_CATALOG)
assert len(SOURCE_TO_CODE) == len(LEAGUES)
assert SOURCE_TO_CODE["GB1"] == "ENG-1"
assert SOURCE_TO_CODE["BRA1"] == "BR-1"
assert SOURCE_TO_CODE["MEXA"] == SOURCE_TO_CODE["MEX1"] == "MEX-1"
assert SOURCE_TO_CODE["COLP"] == SOURCE_TO_CODE["COL1"] == "COL-1"
assert "CLI" not in SOURCE_TO_CODE  # Copa Libertadores
assert "CDLP" not in SOURCE_TO_CODE  # Copa de la Liga Profesional
assert max(SEASONS) == 2026
assert COLLECTION_SCOPE == "minutes-2019-2026-16-leagues-v3"
print(f"League catalog ready: {len(LEAGUE_CODES)} codes and {len(LEAGUES)} sources.")

## **4. Execution**

The notebook runs in `update` mode with no player limit. It completes unfinished historical players, reparses current caches when the season horizon changes, and refreshes active-player performance responses after 24 hours. Profiles and historical pages remain cached.

In [ ]:
RUN_COLLECTION = True
COLLECTION_MODE = 'update'
FORCE_REFRESH = False
REFRESH_INTERVAL_HOURS = 24
MAX_PLAYERS = None

if RUN_COLLECTION:
    collected = collect(
        mode=COLLECTION_MODE,
        force=FORCE_REFRESH,
        refresh_after_hours=REFRESH_INTERVAL_HOURS,
        max_players=MAX_PLAYERS,
        root=ROOT,
    )
    display({name: frame.shape for name, frame in collected.items()})
    display(collected['runs'].tail(1).T.rename(columns={collected['runs'].index[-1]: 'last_run'}))
else:
    print('Set RUN_COLLECTION to True to start the collection.')

## **5. Output audit**

The final check reports the row count of every raw Parquet table. `collection_runs.parquet` records the population selected, network refreshes, cache reparses, errors, and match rows added by each completed run.

In [ ]:
import pandas as pd

for path in sorted((ROOT / 'data' / 'raw').glob('*.parquet')):
    print(path.name, len(pd.read_parquet(path)))

runs_path = ROOT / 'data' / 'raw' / 'collection_runs.parquet'
if runs_path.exists():
    display(pd.read_parquet(runs_path).tail(5))